# 2026 项目 · B · 吉伦特省的住房市场（DVF 2024）

小组：`NN` · 成员：... · 主问题：...

请先阅读 `PROJECT_2026_instructions.ipynb`。这个 notebook 为你提供数据和结构；请替换每一处 *TODO*。

## 数据

- `dvf_gironde_2024.csv.gz` — DGFiP，*Demandes de valeurs foncières géolocalisées*（带地理坐标的房地产交易申报，Licence Ouverte 开放许可）：2024 年
  在吉伦特省（33）登记备案的每一笔房产交易。一行 = 一笔交易的**一部分**（一个分摊单元、一块地、一栋建筑）。
  主要列：`id_mutation`（交易 id）、`date_mutation`、`nature_mutation`（出售、互换……）、`valeur_fonciere`
  （价格，单位 €，**在同一笔交易的每一行上重复出现**）、`type_local`（独栋住宅、公寓、附属建筑、商业用房）、
  `surface_reelle_bati`（m²）、`nombre_pieces_principales`、`surface_terrain`、`nom_commune`、`longitude`、`latitude`。
- 文档：https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/

In [ ]:
from typing import cast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")
from pathlib import Path

# 数据文件夹与 notebook 所在文件夹同级。
DATA = Path("../data") if Path("../data").exists() else Path("data")


def check(condition, ok="回答正确！", hint="还不对，请再读一遍题目。"):
    """打印自检单元格的友好结果。"""
    print(("\u2705 " + ok) if condition else ("\u274c " + hint))


print("环境准备完成。数据文件夹：", DATA.resolve())

In [ ]:
dvf_raw = pd.read_csv(DATA / "dvf_gironde_2024.csv.gz", low_memory=False,
                      dtype={"code_commune": str, "id_parcelle": str})
print(dvf_raw.shape, dvf_raw["id_mutation"].nunique(), "transactions")
dvf_raw.head(3)

### 已知的陷阱（还有别的，自己去找）

- 大多数交易有多行：把 `valeur_fonciere` 按行加总会把同一笔价格重复计算好几次。
- 一笔交易可能同时包括一栋住宅**和**一个车库（`Dépendance`），或者好几套公寓：要判断哪些交易是「简单」且可比的。
- 并非每一行都是出售（`Echange` 互换、`Adjudication` 拍卖、`Expropriation` 征收……）。
- 土地和附属建筑缺少面积；有些价格是象征性的（几欧元），有些则高得离谱。
- 每平方米价格分布非常偏斜：用中位数和对数刻度。

### 可能的问题（也可以自己选）

- 波尔多市内和吉伦特省其他地区的独栋住宅与公寓每平方米价格分别是多少（带置信区间）？
- 什么因素解释了价格：面积、房间数、土地、市镇？（对对数价格做 OLS）
- 2024 年的成交数量或价格是否存在季节性规律？
- 能否用 K-Means 按价格水平和住房类型对市镇进行分组？

## 1. 问题

*TODO：写出主问题和两个子问题。它们为什么有用？对谁有用？*

#### Main question
In 2024, in a simple sale of one dwelling in the Gironde — one transaction, one dwelling, between individuals — how much do the surface, the number of rooms, the land sold with it, the type of dwelling (house or flat), and the address contribute to the price? The address is read twice: the distance to the centre of Bordeaux, and the distance to the two holiday destinations of the department, Arcachon and Lacanau-Océan. In other words: what are 10 m², one room, a house and one kilometre worth?
#### Sub question 1
For flats and for houses separately, what is the effect of +10 m² on the median sale price — in euros and in per cent — with a 95 % confidence interval?
#### Sub question 2
Holding the surface, the number of rooms and the type fixed, by what per cent per kilometre does the price per m² fall with the distance to the centre of Bordeaux, and does that gradient weaken beyond 5 km (a quadratic term, and a hinge at exactly 5 km)?

## Why these questions are useful, and for whom
* For a buyer or a seller: the answer prices a dwelling — ten square metres, one room or one kilometre become a number in euros instead of an opinion. DVF is a public register, so every figure can be checked.
* For the public debate: it separates what the market actually pays for (surface, address) from what it does not (the house premium, land measured but not graded), and it measures how fast the address premium fades once a sale is far from the centre.
* For the group itself: sub-question 2 gives the shape of the address effect (linear? convex? a break at 5 km?), which decides how the models of steps 5 and 6 have to be written.
*  For the next team: the same features, cleaned and documented, are the input of the Machine-Learning course of semester 2; a written question tells them what the table is for.


#### 主要研究问题
2024年，在法国吉伦特省（Gironde），对于个人之间一笔仅涉及一套住宅的简单买卖交易，住宅面积、房间数量、随房出售的土地面积、住宅类型（独立屋或公寓）以及地址，各自对成交价格有多大影响？
地址从两个角度衡量：一是到波尔多市中心的距离，二是到省内两个度假目的地——阿卡雄（Arcachon）和拉卡诺海滨（Lacanau-Océan）——的距离。换句话说：10平方米、一个房间、独立屋这一住宅类型，以及一公里的距离，各值多少钱？
#### 子问题1
分别针对公寓和独立屋，住宅面积增加10平方米，对成交价格中位数有何影响？请以欧元金额和百分比表示，并给出95%置信区间。
#### 子问题2
在住宅面积、房间数量和住宅类型保持不变的条件下，到波尔多市中心的距离每增加一公里，每平方米价格下降百分之几？这一价格梯度是否在距离超过5公里后减弱？通过加入距离的二次项，以及在恰好5公里处设置折点的分段线性项来检验。
## 这些问题为何有用，以及对谁有用
* 对买家或卖家而言：研究结果可以为住宅特征定价，让“10平方米”“一个房间”或“一公里”从主观判断变成具体的欧元金额。DVF（法国房地产成交价值数据库）是公开登记数据，因此每个数字都可以核查。
* 对公共讨论而言：研究可以区分市场实际愿意为哪些因素付费（面积、地址），以及哪些因素未体现出相应溢价（独立屋溢价、仅衡量面积而未区分品质的土地），并衡量住宅距离市中心越远，地址溢价消退得有多快。
* 对本小组而言：子问题2揭示地址效应的具体形态：是线性的、凸性的，还是在5公里处出现转折？这将决定第5步和第6步的模型应如何设定。
* 对下一组团队而言：经过清洗并附有说明的同一组特征，将作为第二学期机器学习课程的输入数据；明确写出的研究问题，可以让接手团队理解这张数据表的用途。

## 2. 数据审查

*TODO：一行代表什么？形状、类型、缺失值、重复值、不可能的值、覆盖的时段。复用 TP3 A 部分的 `quality_report`。*

In [ ]:
# 【复用 TP3A】审计第一张表：行数、交易笔数、列数。
n_rows = len(dvf_raw)
n_sales = dvf_raw["id_mutation"].nunique()
per_sale = dvf_raw.groupby("id_mutation").size()

print(f"rows = {n_rows:,}  |  transactions = {n_sales:,}  |  rows per transaction = {n_rows / n_sales:.2f}")
print("\nrows per transaction (number of parts in the same sale):")
print(per_sale.value_counts().sort_index().head(8).to_string())
print("\ndistribution of the number of rows per sale:")
print(per_sale.describe().round(1).to_string())

# 价格在同一笔交易的每一行都重复：直接求和会把价格乘上行数。
assert (dvf_raw.groupby("id_mutation")["valeur_fonciere"].nunique() <= 1).all()
assert per_sale.max() > 10

In [ ]:
def quality_report(df):
    # 【复用 TP3A】审计表：类型、缺失率、唯一值数、一个样例。
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_pct": (100 * df.isna().mean()).round(1),
        "n_unique": df.nunique(),
        "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })


report = quality_report(dvf_raw).sort_values("missing_pct", ascending=False)
report

In [ ]:
# 【审计】交易性质的六种取值与行数：只有 Vente 是我们要的市场价。
print("nature of the transaction")
print(dvf_raw["nature_mutation"].value_counts(dropna=False).to_string())

print("\ntype of local (NaN = a plot with no building)")
print(dvf_raw["type_local"].value_counts(dropna=False).to_string())

print("\nfirst and last date registered:", dvf_raw["date_mutation"].min(), "->", dvf_raw["date_mutation"].max())

In [ ]:
exact_dup = int(dvf_raw.duplicated().sum())
same_parcel = int(dvf_raw.duplicated(["id_mutation", "id_parcelle"]).sum())
print(f"|{'exact duplicate rows:':50}| {exact_dup:,}")
print(f"|{'rows sharing (id_mutation, id_parcelle), not dupl.':50}| {same_parcel:,}")

# -- 不可能或可疑的取值 ---------------------------------------------------
print(f"|{'prices <= 10 EUR:':50}| {int(dvf_raw['valeur_fonciere'].le(10).sum()):,}")
print(f"|{'prices > 5 M EUR:':50}| {int(dvf_raw['valeur_fonciere'].gt(5e6).sum()):,}")
print(f"|{'missing price:':50}| {int(dvf_raw['valeur_fonciere'].isna().sum()):,}")
print(f"|{'surface_reelle_bati <= 0 :':50}| {int(dvf_raw['surface_reelle_bati'].le(0).sum()):,}")
print(f"|{'rows without coordinates:':50}| {int(dvf_raw[['longitude', 'latitude']].isna().any(axis=1).sum()):,}")

print("\nprice (EUR):")
print(dvf_raw["valeur_fonciere"].describe(percentiles=[.05, .5, .95]).round(0).to_string())



## 3. 清洗

*TODO：写一个函数 `clean_...(raw)`，把决定写在注释里，之后用 `assert` 检查。*

In [ ]:
# 【常量】Bordeaux Métropole 的 28 个 INSEE 代码（只在稳健性检验里用）。
# Bordeaux Métropole = 28 个市镇（INSEE 代码），来源：https://comersis.fr/epci.php?epci=243300316
METRO_CODES = {
    "33003", "33004", "33013", "33032", "33039", "33056", "33063", "33065", "33075", "33096",
    "33119", "33162", "33167", "33192", "33069", "33200", "33519", "33249", "33273", "33281",
    "33312", "33318", "33376", "33434", "33449", "33487", "33522", "33550",
}
BORDEAUX = "33063"
HABITATION = ["Maison", "Appartement"]
LAT_MIN, LAT_MAX = 44.0, 45.7  # the Gironde in a slightly wider box
LON_MIN, LON_MAX = -1.4, 0.5
PRICE_MIN, PRICE_MAX = 1_000, 5_000_000
PM2_MIN, PM2_MAX = 300, 20_000


def clean_dvf(raw, verbose=True):
    df = raw.copy()
    rows_in = len(df)
    removed = []

    def keep_where(mask, reason):
        nonlocal df
        removed.append((reason, int((~mask).sum())))
        df = df[mask]

    # 1：文件里日期是字符串，转成日期并只保留 2024 年（无缺失日期）。
    df["date_mutation"] = pd.to_datetime(df["date_mutation"], errors="coerce")
    keep_where(df["date_mutation"].notna(), "date is not a date")
    keep_where(df["date_mutation"].dt.year == 2024, "date outside 2024")

    # 2：只有普通买卖才是市场价；期房（尚未建成）、置换、拍卖、征收都剔除。
    keep_where(df["nature_mutation"].eq("Vente"), "not a plain Vente")

    # 3：完全相同的行不含新信息，会让同一观测被算两次。
    keep_where(~df.duplicated(), "exact duplicate row")

    # 4：象征性价格（≤ 10 €）不是市场价；超过 500 万 € 的多是整栋或商业地产，不是一套住宅。
    keep_where(df["valeur_fonciere"].notna(), "price missing")
    keep_where(df["valeur_fonciere"].ge(PRICE_MIN), "symbolic price (< 1,000 EUR)")
    keep_where(df["valeur_fonciere"].le(PRICE_MAX), "estate / commercial price (> 5 M EUR)")

    # 5：位置要用于行政区与距离，所以必须存在且落在吉伦特省框内。
    keep_where(df["latitude"].notna() & df["longitude"].notna(), "D5 coordinates missing")
    keep_where(df["latitude"].between(LAT_MIN, LAT_MAX) & df["longitude"].between(LON_MIN, LON_MAX),
               "coordinates outside the Gironde")

    # 6：研究对象是住宅，因此这笔交易至少要含一套公寓或独栋。
    keep_where(df["id_mutation"].isin(df.loc[df["type_local"].isin(HABITATION), "id_mutation"]),
               "sale without a house or a flat")

    # 7：简单交易 = 恰好一套住宅（房型与面积在住宅行中唯一）
    # （同一套住宅可能因两种土地性质被写两次）；其余行只能是附属物
    # （Dépendance）或纯土地（无房型、无建筑面积）。
    hab = df[df["type_local"].isin(HABITATION)].drop_duplicates(
        ["id_mutation", "type_local", "surface_reelle_bati"])
    signature = hab.groupby("id_mutation").agg(n_type=("type_local", "nunique"),
                                               n_surface=("surface_reelle_bati", "nunique"))
    simple = signature[(signature["n_type"] == 1) & (signature["n_surface"] == 1)].index
    keep_where(df["id_mutation"].isin(simple), "sale with several dwellings")

    other = df[~df["type_local"].isin(HABITATION)]
    tolerated = other["type_local"].eq("Dépendance") | (
            other["type_local"].isna() & other["surface_reelle_bati"].isna())
    keep_where(~df["id_mutation"].isin(other.loc[~tolerated, "id_mutation"]),
               "sale mixed with a commercial or industrial local")

    # 8：住宅；价格是整笔交易的价格（在每一行重复）。
    hab = df[df["type_local"].isin(HABITATION)]
    sales = hab.groupby("id_mutation").agg(
        date_mutation=("date_mutation", "first"),
        valeur_fonciere=("valeur_fonciere", "first"),
        type_local=("type_local", "first"),
        surface_reelle_bati=("surface_reelle_bati", "first"),
        nombre_pieces_principales=("nombre_pieces_principales", "max"),
        nom_commune=("nom_commune", "first"),
        code_commune=("code_commune", "first"),
        code_postal=("code_postal", "first"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
    )
    # 8：土地：先按（地块, 土地性质）取最大值，再在这笔交易内跨地块求和。
    land_rows = df[["id_mutation", "id_parcelle", "nature_culture", "surface_terrain"]].dropna(
        subset=["surface_terrain"]).drop_duplicates()
    land = (land_rows.groupby(["id_mutation", "id_parcelle", "nature_culture"], dropna=False)["surface_terrain"]
            .max().groupby(level="id_mutation").sum())
    dep = df[df["type_local"].eq("Dépendance")].drop_duplicates(
        ["id_mutation", "id_parcelle", "surface_reelle_bati"])
    n_dep = dep.groupby("id_mutation").size()

    sales["code_postal"] = sales["code_postal"].astype("Int64")  # 33290, not 33290.0
    sales["n_dependances"] = n_dep.reindex(sales.index).fillna(0).astype(int)
    sales["surface_terrain_total"] = land.reindex(sales.index)
    sales["nombre_pieces_principales"] = sales["nombre_pieces_principales"].where(
        sales["nombre_pieces_principales"] > 0)  # 0 room with a surface is not a real value

    # 9：算出每 m² 价格，再用一个合理区间筛掉明显错误。
    n_before = len(sales)
    sales = sales[sales["surface_reelle_bati"].gt(0)]  # no division by zero
    removed_sales = [("D9 surface_reelle_bati <= 0", n_before - len(sales))]

    sales["prix_m2"] = sales["valeur_fonciere"] / sales["surface_reelle_bati"]
    sales["log_prix"] = np.log(sales["valeur_fonciere"])
    sales["log_prix_m2"] = np.log(sales["prix_m2"])
    n_before = len(sales)
    sales = sales[sales["prix_m2"].between(PM2_MIN, PM2_MAX)]
    removed_sales.append(("prix_m2 outside [300, 20,000] EUR/m2", n_before - len(sales)))

    # -- 三个行政区（INSEE 代码稳定，市镇名字会变） ---------------------------------
    sales["zone"] = np.where(sales["code_commune"].eq(BORDEAUX), "Bordeaux",
                             np.where(sales["code_commune"].isin(METRO_CODES), "Métropole",
                                      "Reste de la Gironde"))
    sales["month"] = sales["date_mutation"].dt.month

    if verbose:
        print(f"rows kept: {len(df):,} of {rows_in:,}")
        print(pd.DataFrame(removed, columns=["row filter", "rows removed"])
              .set_index("row filter").to_string())
        print("\nsales removed after aggregation:")
        print(pd.DataFrame(removed_sales, columns=["sale filter", "sales removed"])
              .set_index("sale filter").to_string())
        print(f"\nsimple sales kept: {len(sales):,} of {raw['id_mutation'].nunique():,} transactions")

    return sales.reset_index()


############################################################################################################
dvf = clean_dvf(dvf_raw)
dvf.head(3)


#### CHECK THE RESULT 检查结果

In [ ]:
assert dvf["id_mutation"].is_unique  # one row = one sale
assert set(dvf["type_local"]) == set(HABITATION)  # a house or a flat, nothing else
assert dvf["surface_reelle_bati"].gt(0).all()  # no zero surface before dividing
assert dvf["valeur_fonciere"].ge(PRICE_MIN).all()
assert dvf["valeur_fonciere"].le(PRICE_MAX).all()
assert dvf["date_mutation"].dt.year.eq(2024).all()
assert dvf["latitude"].between(LAT_MIN, LAT_MAX).all()
assert dvf["longitude"].between(LON_MIN, LON_MAX).all()
assert dvf["prix_m2"].between(PM2_MIN, PM2_MAX).all()
assert set(dvf["zone"]) == {"Bordeaux", "Métropole", "Reste de la Gironde"}
assert dvf[["valeur_fonciere", "surface_reelle_bati", "type_local", "zone", "date_mutation"]].notna().all().all()
assert dvf["n_dependances"].ge(0).all()

print(f"{len(dvf):,} simple sales | {len(dvf) / dvf_raw['id_mutation'].nunique():.0%} of the transactions of the file")
print(
    f"median price {dvf['valeur_fonciere'].median():,.0f} EUR | median surface {dvf['surface_reelle_bati'].median():.0f} m2")
dvf[["valeur_fonciere", "surface_reelle_bati", "nombre_pieces_principales", "surface_terrain_total",
     "n_dependances", "prix_m2"]].describe(percentiles=[.5, .95]).round(1)

## 4. 探索

*TODO：至少五张图，每张图后面跟一句话。*

In [ ]:
# 【复用 TP2A】特征工程：对数面积、土地，以及到佩贝朗广场的向量化 haversine 距离；N_BOOT 也在这里。
FIG = Path("figures")
FIG.mkdir(exist_ok=True)
N_BOOT = 1000

# -- 住宅 ---------------------------------------------------------------------
dvf["type_local"] = dvf["type_local"].astype(str)
dvf["is_maison"] = dvf["type_local"].eq("Maison").astype(int)
dvf["log_surface"] = np.log(dvf["surface_reelle_bati"])
dvf["land_m2"] = dvf["surface_terrain_total"].fillna(0)
dvf["has_land"] = dvf["surface_terrain_total"].notna().astype(int)
dvf["log_land"] = np.log1p(dvf["land_m2"])

# -- 把地址压缩成一个数：到波尔多市中心的距离 -----------------------------------
# 中心点 = 佩贝朗广场（市政厅），北纬 44.8378 / 东经 −0.5792。
# 距离 = 大圆距离（haversine，单位 km）：忽略道路、桥梁与交通，这是局限
# （第 8 步写明）；但这是文件唯一能支撑的距离，而且每一笔都能算。
CENTRE_LAT, CENTRE_LON, EARTH_KM = 44.8378, -0.5792, 6371.0


def haversine_km(lat, lon, lat0=CENTRE_LAT, lon0=CENTRE_LON):
    # 球面两点的大圆距离：2R·arcsin(√a)。
    lat, lon = np.radians(lat), np.radians(lon)
    lat0, lon0 = np.radians(lat0), np.radians(lon0)
    a = np.sin((lat - lat0) / 2) ** 2 + np.cos(lat) * np.cos(lat0) * np.sin((lon - lon0) / 2) ** 2
    return 2 * EARTH_KM * np.atan(np.sqrt(a), np.sqrt(1 - a))


dvf["distance_km"] = haversine_km(dvf["latitude"].to_numpy(), dvf["longitude"].to_numpy())
dvf["distance_km_sq"] = dvf["distance_km"] ** 2
dvf["km_beyond_5"] = (dvf["distance_km"] - 5).clip(lower=0)  # hinge: 0 inside 5 km, km past it after

assert dvf["distance_km"].between(0, 120).all()  # a sale in the Gironde is within 120 km of Pey Berland
assert dvf[["log_surface", "distance_km", "log_prix"]].notna().all().all()
print(f"{len(dvf):,} simple sales of 2024")
print(f"distance to the centre: median {dvf['distance_km'].median():.1f} km"
      f" | 50 % of the sales between {dvf['distance_km'].quantile(0.25):.1f} and {dvf['distance_km'].quantile(0.75):.1f} km"
      f" | 95 % under {dvf['distance_km'].quantile(0.95):.0f} km | max {dvf['distance_km'].max():.0f} km")
print(f"houses {dvf['is_maison'].mean():.0%} | flats {1 - dvf['is_maison'].mean():.0%}"
      f" | with land recorded {dvf['has_land'].mean():.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, kind in zip(axes, ["Appartement", "Maison"]):
    x = dvf.loc[dvf["type_local"].eq(kind), "prix_m2"]
    bins = np.geomspace(300, x.quantile(0.999), 45)
    ax.hist(x, bins=bins, color="#4C72B0" if kind == "Appartement" else "#DD8452", alpha=0.85)
    ax.set_xscale("log")
    ax.axvline(x.median(), color="black", lw=1.6, label=f"median {x.median():,.0f} EUR/m2")
    ax.axvline(x.mean(), color="black", lw=1.2, ls="--", label=f"mean {x.mean():,.0f} EUR/m2")
    ax.set_title(f"{kind} ({len(x):,} sales)")
    ax.set_xlabel("price per m2 (EUR, log scale)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("number of sales")
fig.suptitle("A long right tail: the mean is above the median in both markets", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "fig1_pm2_distribution.png", dpi=150, bbox_inches="tight")

In [ ]:
# 【复用 TP3B】图 2：每 m² 中位价随距离的变化（按房型分线）+ 每个距离带的成交量。
edges = [0, 2, 4, 6, 8, 12, 16, 20, 25, 30, 40, 200]
d = dvf.copy()
d["dist_bin"] = pd.cut(d["distance_km"], edges)
tab = d.pivot_table(index="dist_bin", columns="type_local", values="prix_m2", aggfunc="median", observed=True)
cnt = d.pivot_table(index="dist_bin", columns="type_local", values="prix_m2", aggfunc="size", observed=True)
shown = tab.where(cnt >= 20)  # a bin with fewer than 20 sales says nothing
x = [(iv.left + iv.right) / 2 for iv in tab.index]
x[-1] = 55.0  # the open top bin
labels = [f"{int(iv.left)}-{int(iv.right)}" if iv.right < 200 else f"{int(iv.left)}+" for iv in tab.index]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), gridspec_kw={"width_ratios": [1.55, 1]})
ax = axes[0]
for kind, color in [("Appartement", "#4C72B0"), ("Maison", "#DD8452")]:
    ax.plot(x, shown[kind].values, marker="o", color=color, label=kind)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, fontsize=7.5)
ax.set_yscale("log")
ax.set_xlabel("distance to the centre of Bordeaux (km)")
ax.set_ylabel("median price per m2 (EUR, log scale)")
ax.set_title("Price per m2 against the distance to the centre")
ax.legend(fontsize=8)

bx = np.arange(len(labels))
axes[1].bar(bx - 0.2, cnt["Appartement"].fillna(0).values, width=0.4, color="#4C72B0", label="Appartement")
axes[1].bar(bx + 0.2, cnt["Maison"].fillna(0).values, width=0.4, color="#DD8452", label="Maison")
axes[1].set_xticks(bx)
axes[1].set_xticklabels(labels, rotation=45, fontsize=7.5)
axes[1].set_yscale("log")
axes[1].set_xlabel("distance to the centre (km)")
axes[1].set_ylabel("number of sales (log scale)")
axes[1].set_title("Where the sales are")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "fig2_distance_profile.png", dpi=150, bbox_inches="tight")
print((shown.round(0).to_string()))

In [ ]:
# 【复用 TP3B】图 3：双对数坐标下的价格–面积关系（斜率 ≈ 0.7–0.8）。
fig, ax = plt.subplots(figsize=(7.2, 5.2))
for kind, color in [("Appartement", "#4C72B0"), ("Maison", "#DD8452")]:
    s = dvf[dvf["type_local"].eq(kind)]
    ax.scatter(s["surface_reelle_bati"], s["valeur_fonciere"], s=6, alpha=0.25, color=color, label=kind)
    grid = np.quantile(s["surface_reelle_bati"], np.linspace(0.05, 0.95, 12))
    med = s.groupby(pd.cut(s["surface_reelle_bati"], np.r_[0, grid, np.inf]), observed=True)["valeur_fonciere"].median()
    ax.plot([iv.mid for iv in med.index], med.values, color=color, lw=2.4, marker="o", ms=3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("surface (m2, log scale)")
ax.set_ylabel("sale price (EUR, log scale)")
ax.set_title("Price against surface: a straight line on a log-log plane")
ax.legend()
fig.tight_layout()
fig.savefig(FIG / "fig3_surface_price.png", dpi=150, bbox_inches="tight")

In [ ]:
# 【复用 TP3B】图 5：以佩贝朗广场为中心的地图，颜色 = 每 m² 价，环 = 5 / 15 km。
# 以南佩贝朗广场为原点（东西 / 南北，单位 km）重新画的地图：环就是模型用的距离。
lat0, lon0 = CENTRE_LAT, CENTRE_LON
kx = (dvf["longitude"] - lon0) * 111.32 * np.cos(np.radians(lat0))
ky = (dvf["latitude"] - lat0) * 110.57
fig, ax = plt.subplots(figsize=(7.8, 6.4))
sc = ax.scatter(kx, ky, c=dvf["prix_m2"], s=5, alpha=0.35, cmap="magma_r", vmin=1000, vmax=7000, label="sale")
pl = plt.colorbar(sc, ax=ax, shrink=0.9)
pl.set_label("price per m2 (EUR)")
for r in [5, 10, 20, 40]:
    ax.add_patch(plt.Circle((0, 0), r, fill=False, color="#333333", lw=0.9, ls="--"))
    ax.text(r * 0.71, r * 0.71, f"{r} km", fontsize=8, color="#333333", ha="center", va="center")
ax.plot(0, 0, marker="*", color="black", ms=14)
ax.text(0.6, -2.2, "Place Pey Berland", fontsize=8)
ax.set_aspect("equal")
ax.set_xlabel("km east of the centre")
ax.set_ylabel("km north of the centre")
ax.set_title("The Gironde around Bordeaux: the dashed rings are the distance used in the model")
fig.tight_layout()
fig.savefig(FIG / "fig5_map_distance.png", dpi=150, bbox_inches="tight")

## 5. 统计

#### 1.按距离带给出每 m² 中位价的 95 % 区间

In [ ]:
# 【复用 TP3C】5.1：bootstrap 中位数区间；均值区间用课程给的 DescrStatsW.tconfint_mean()。
def bootstrap_median_ci(x, n_boot=N_BOOT, seed=0):
    # 中位数的 95 % 区间：有放回重抽样，取重抽样中位数的 2.5 % 与 97.5 % 分位。
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    meds = np.empty(n_boot)
    for i in range(n_boot):
        meds[i] = np.median(rng.choice(x, size=x.size, replace=True))
    return np.median(x), np.percentile(meds, 2.5), np.percentile(meds, 97.5)


BANDS = [0, 5, 10, 20, 40, 200]


def band_name(iv):
    return f"{int(iv.left)}-{int(iv.right)} km" if iv.right < 200 else f"{int(iv.left)} km +"


d = dvf.copy()
d["band"] = pd.cut(d["distance_km"], BANDS, right=False)

rows = []
for (band, kind), sub in d.groupby(["band", "type_local"], observed=True):
    pm2 = sub["prix_m2"]
    n = len(pm2)
    med, lo, hi = bootstrap_median_ci(pm2)
    lo_m, hi_m = sm.stats.DescrStatsW(pm2).tconfint_mean()  # TP3 C: textbook interval of the MEAN
    half = (hi_m - lo_m) / 2
    rows.append({"band": band_name(band), "type": kind, "n": n, "median": med, "ci_low": lo, "ci_high": hi,
                 "plus_minus_pct": 100 * (hi - lo) / med / 2, "mean_half_ci": half})

ci_table = pd.DataFrame(rows)
show = ci_table.assign(
    median=ci_table["median"].round(0),
    ci=[f"[{a:,.0f}; {b:,.0f}]" for a, b in zip(ci_table.ci_low, ci_table.ci_high)],
    plus_minus_pct=ci_table["plus_minus_pct"].round(1),
    mean_half_ci=ci_table["mean_half_ci"].round(0),
)[["band", "type", "n", "median", "ci", "plus_minus_pct", "mean_half_ci"]]
print(show.to_string(index=False))
print("\nThe bootstrap interval is narrow (a few percent of the median) and asymmetric; the t interval\n" +
      "on the same rows is wider and centred on the mean, which the expensive tail pushes above the median.\n"
      "Neighbouring bands overlap, so the table says 'the level falls with the distance', not 'each band is distinct'.")

#### MAIN 价格特征占比

In [ ]:
# 【复用 第六讲 + TP3C】5.4：四个嵌套的特征价格 OLS，打印系数与 R² 阶梯；读数助手 as_pct 也在这里。
DWELLING = "log_surface + nombre_pieces_principales + C(type_local) + log_land + has_land"
h0 = smf.ols(f"log_prix ~ {DWELLING}", data=dml).fit()
h1 = smf.ols(f"log_prix ~ {DWELLING} + distance_km", data=dml).fit()
h2 = smf.ols(f"log_prix ~ {DWELLING} + distance_km + C(month)", data=dml).fit()
h3 = smf.ols(f"log_prix ~ {DWELLING} + distance_km + C(month) + C(zone)", data=dml).fit()
h = h2  # the model reported below

MED_PRICE = dml["valeur_fonciere"].median()
MED_SURFACE = dml["surface_reelle_bati"].median()

LABELS = {
    "log_surface": ("log(surface)", "x 10 % surface", "elastic"),
    "nombre_pieces_principales": ("rooms", "+1 room", "level"),
    "C(type_local)[T.Maison]": ("house (vs flat)", "house vs flat", "level"),
    "log_land": ("log(land)", "x 10 % land", "elastic"),
    "has_land": ("land recorded", "0 -> 1", "level"),
    "distance_km": ("distance (km)", "+1 km", "level"),
}


def as_pct(x, kind):
    # log-log 系数是弹性：输入多 10 % 对应 1.1^b − 1，而不是 exp(b) − 1。
    return 100 * ((1.1 ** x - 1) if kind == "elastic" else (np.exp(x) - 1))


rows = []
for term, (name, unit, kind) in LABELS.items():
    b = h.params[term]
    lo, hi = h.conf_int().loc[term]
    pct = as_pct(b, kind)
    rows.append({
        "feature": name, "unit": unit, "coef": b, "p": h.pvalues[term],
        "pct": pct,
        "pct_lo": as_pct(lo, kind), "pct_hi": as_pct(hi, kind),
        "EUR": MED_PRICE * pct / 100,
        "EUR_lo": MED_PRICE * as_pct(lo, kind) / 100, "EUR_hi": MED_PRICE * as_pct(hi, kind) / 100,
    })

# 把面积系数读成子问题 1 的单位：中位面积上“多 10 m²”。
bS = h.params["log_surface"]
loS, hiS = h.conf_int().loc["log_surface"]
p10 = 100 * ((1 + 10 / MED_SURFACE) ** bS - 1)
p10_lo = 100 * ((1 + 10 / MED_SURFACE) ** loS - 1)
p10_hi = 100 * ((1 + 10 / MED_SURFACE) ** hiS - 1)
rows.append({
    "feature": "surface, +10 m2 at the median", "unit": f"{MED_SURFACE:.0f} -> {MED_SURFACE + 10:.0f} m2",
    "coef": bS, "p": h.pvalues["log_surface"],
    "pct": p10, "pct_lo": p10_lo, "pct_hi": p10_hi,
    "EUR": MED_PRICE * p10 / 100,
    "EUR_lo": MED_PRICE * p10_lo / 100, "EUR_hi": MED_PRICE * p10_hi / 100,
})

hedonic = pd.DataFrame(rows)
print(f"n = {int(h.nobs):,} simple sales | median sale {MED_PRICE:,.0f} EUR | median surface {MED_SURFACE:.0f} m2\n")
print(f"{'feature':30s} {'unit':14s} {'% effect':>10s} {'95 % interval':>22s} {'EUR at median':>16s}")
for r in hedonic.itertuples():
    print(f"{r.feature:30s} {r.unit:14s} {r.pct:+10.1f}   [{r.pct_lo:+6.1f}; {r.pct_hi:+6.1f}]   "
          f"{r.EUR:+9,.0f}   [{r.EUR_lo:,.0f}; {r.EUR_hi:,.0f}]")
print(f"\n{'':30s} {'':14s} {'':>10s} {'':>22s}   * the median price is {MED_PRICE:,.0f} EUR")
print("\nR2 on the log price:")
print(f"  dwelling only            {h0.rsquared:.3f}")
print(f"  + distance               {h1.rsquared:.3f}   (+{100 * (h1.rsquared - h0.rsquared):.1f} points)")
print(f"  + month                  {h2.rsquared:.3f}   (+{100 * (h2.rsquared - h1.rsquared):.2f} points)")
print(f"  + the three zones        {h3.rsquared:.3f}   (+{100 * (h3.rsquared - h2.rsquared):.2f} points)")

#### Sub 1 每多 10 m² 值多少

In [ ]:
# 【复用 第六讲 + TP3C】5.2：markup_10m2 —— 中位数分位回归给欧元，log-log 弹性给百分比。
def markup_10m2(kind):
    # 一个市场的“多 10 m² 值多少”：欧元来自中位数分位回归，百分比来自 log-log 弹性。
    s = dvf[dvf["type_local"].eq(kind)]
    qr = smf.quantreg("valeur_fonciere ~ surface_reelle_bati", s).fit(q=0.5)
    slope = qr.params["surface_reelle_bati"]
    slope_lo, slope_hi = qr.conf_int().loc["surface_reelle_bati"]

    el = smf.ols("log_prix ~ log_surface", s).fit()
    e = el.params["log_surface"]
    e_lo, e_hi = el.conf_int().loc["log_surface"]

    S = s["surface_reelle_bati"].median()
    pct = lambda b: 100 * ((1 + 10 / S) ** b - 1)  # +10 m2 at the median surface, in %
    return {
        "type": kind,
        "n": len(s),
        "median_price": s["valeur_fonciere"].median(),
        "median_surface": S,
        "EUR_per_10m2": 10 * slope,
        "EUR_lo": 10 * slope_lo,
        "EUR_hi": 10 * slope_hi,
        "elasticity": e,
        "pct_per_10m2": pct(e),
        "pct_lo": pct(e_lo),
        "pct_hi": pct(e_hi),
    }


ten_m2 = pd.DataFrame([markup_10m2(k) for k in ["Appartement", "Maison"]])
for r in ten_m2.itertuples():
    print(f"{r.type:12s} n = {r.n:,}   median sale {r.median_price:,.0f} EUR for {r.median_surface:.0f} m2")
    print(f"    +10 m2  ->  {r.EUR_per_10m2:+,.0f} EUR   95 % [{r.EUR_lo:,.0f}; {r.EUR_hi:,.0f}]"
          f"   (median regression)")
    print(f"                 {r.pct_per_10m2:+.1f} %     95 % [{r.pct_lo:+.1f}; {r.pct_hi:+.1f}]"
          f"   (elasticity {r.elasticity:.2f})")
    print()

#### Sub 2 距离和价格的关系

In [ ]:
# 【复用 第六讲 + TP3C】5.3：B1 线性、B2 二次项、B3 5 km 折点，以及两个子样本的稳健性回归。
dml = dvf.dropna(subset=["nombre_pieces_principales"]).copy()
dml["type_local"] = pd.Categorical(dml["type_local"], categories=["Appartement", "Maison"])
dml["zone"] = pd.Categorical(dml["zone"], categories=["Reste de la Gironde", "Métropole", "Bordeaux"])
CTRL = "log_surface + nombre_pieces_principales + C(type_local) + log_land + has_land"

b1 = smf.ols(f"log_prix_m2 ~ {CTRL} + distance_km", data=dml).fit()
b2 = smf.ols(f"log_prix_m2 ~ {CTRL} + distance_km + distance_km_sq", data=dml).fit()
b3 = smf.ols(f"log_prix_m2 ~ {CTRL} + distance_km + km_beyond_5", data=dml).fit()


def pct_coef(fit, term):
    # 把对数模型（log-level 或 log-log）的系数读成百分比变化，并给出 95 % 区间。
    b = fit.params[term]
    lo, hi = fit.conf_int().loc[term]
    return 100 * (np.exp(b) - 1), 100 * (np.exp(lo) - 1), 100 * (np.exp(hi) - 1), fit.pvalues[term]


p, plo, phi, pv = pct_coef(b1, "distance_km")
print(f"B1  one km further: {p:+.2f} % per m2   [{plo:+.2f}; {phi:+.2f}]   p = {pv:.1e}   R2 = {b1.rsquared:.3f}")

q = b2.params["distance_km_sq"]
qlo, qhi = b2.conf_int().loc["distance_km_sq"]
print(f"B2  + distance^2: {q:+.3e} per km^2   [{qlo:+.3e}; {qhi:+.3e}]   p = {b2.pvalues['distance_km_sq']:.1e}"
      f"   R2 = {b2.rsquared:.3f}")
print(
    f"    the parabola turns up at {b2.params['distance_km'] / (-2 * q):.0f} km: beyond it, only the hinge is trustworthy")
cov = b2.cov_params()
print("    the marginal gradient, percent of the price per m2 per km:")
for dd in [1, 5, 10, 20, 30]:
    me = b2.params["distance_km"] + 2 * q * dd
    var = (cov.loc["distance_km", "distance_km"] + 4 * dd ** 2 * cov.loc["distance_km_sq", "distance_km_sq"]
           + 4 * dd * cov.loc["distance_km", "distance_km_sq"])
    se = np.sqrt(var)
    print(f"      at {dd:2d} km: {100 * (np.exp(me) - 1):+.2f} %/km"
          f"   95 % [{100 * (np.exp(me - 1.96 * se) - 1):+.2f}; {100 * (np.exp(me + 1.96 * se) - 1):+.2f}]")

b_h = b3.params["km_beyond_5"]
inside = 100 * (np.exp(b3.params["distance_km"]) - 1)
slope_out = b3.params["distance_km"] + b_h
V3 = b3.cov_params()
se_out = np.sqrt(V3.loc["distance_km", "distance_km"] + V3.loc["km_beyond_5", "km_beyond_5"]
                 + 2 * V3.loc["distance_km", "km_beyond_5"])
out_pct = 100 * (np.exp(slope_out) - 1)
print(f"\nB3  hinge at 5 km: {inside:+.2f} %/km under 5 km, {out_pct:+.2f} %/km beyond"
      f"   95 % [{100 * (np.exp(slope_out - 1.96 * se_out) - 1):+.2f}; {100 * (np.exp(slope_out + 1.96 * se_out) - 1):+.2f}]")
print(
    f"    the slope changes by {out_pct - inside:+.2f} points of %/km after the knot   p = {b3.pvalues['km_beyond_5']:.1e}"
    f"   R2 = {b3.rsquared:.3f}")

print("\nsplit samples, same controls, one slope each side of 5 km:")
for name, sub in [("under 5 km", dml[dml["distance_km"].lt(5)]), ("5 km and beyond", dml[dml["distance_km"].ge(5)])]:
    m = smf.ols(f"log_prix_m2 ~ {CTRL} + distance_km", sub).fit()
    a, alo, ahi, ap = pct_coef(m, "distance_km")
    print(f"  {name:15s} n = {int(m.nobs):,}   {a:+.2f} %/km   [{alo:+.2f}; {ahi:+.2f}]")

b4 = smf.ols(f"log_prix_m2 ~ {CTRL} + distance_km + distance_km_sq + C(zone)", data=dml).fit()
print(f"\nB4  B2 + the three administrative zones: R2 {b2.rsquared:.3f} -> {b4.rsquared:.3f}")
for t, name in [("C(zone)[T.Métropole]", "Métropole"), ("C(zone)[T.Bordeaux]", "Bordeaux")]:
    a, alo, ahi, ap = pct_coef(b4, t)
    print(f"    {name:10s} {a:+.1f} % per m2 vs the rest of the Gironde   [{alo:+.1f}; {ahi:+.1f}]   p = {ap:.1e}")

In [ ]:
# 【复用 第六讲 + TP3B】图 7：折点模型 B3 的拟合曲线 vs 原始分箱中位数。
def gradient_curve(kind, grid, fit=b3):
    # 合并样本的折点模型 B3，在某一房型的“中位住宅”上取值：即系数隐含的那条曲线。
    s = dml[dml["type_local"].eq(kind)]
    base = pd.DataFrame({
        "log_surface": np.full_like(grid, s["log_surface"].median()),
        "nombre_pieces_principales": np.full_like(grid, s["nombre_pieces_principales"].median()),
        "log_land": np.full_like(grid, s["log_land"].median()),
        "has_land": np.full_like(grid, s["has_land"].median()),
        "type_local": pd.Categorical(np.repeat(kind, grid.size), categories=["Appartement", "Maison"]),
        "distance_km": grid,
        "km_beyond_5": np.maximum(grid - 5, 0),
    })
    return np.exp(fit.predict(base))


edges = [0, 2, 4, 6, 8, 12, 16, 20, 25, 30, 40, 200]
d = dvf.copy()
d["dist_bin"] = pd.cut(d["distance_km"], edges)
tab = d.pivot_table(index="dist_bin", columns="type_local", values="prix_m2", aggfunc="median", observed=True)
cnt = d.pivot_table(index="dist_bin", columns="type_local", values="prix_m2", aggfunc="size", observed=True)
tab = tab.where(cnt >= 20)
x = [(iv.left + iv.right) / 2 for iv in tab.index]
x[-1] = 55.0
grid = np.linspace(0.5, 55, 250)

fig, ax = plt.subplots(figsize=(9.5, 5.0))
for kind, color in [("Appartement", "#4C72B0"), ("Maison", "#DD8452")]:
    ax.scatter(x, tab[kind].values, color=color, s=30, zorder=3, label=f"{kind}: raw medians")
    ax.plot(grid, gradient_curve(kind, grid), color=color, lw=2.2, label=f"{kind}: hinge B3, median dwelling")
ax.set_yscale("log")
ax.set_xlabel("distance to the centre of Bordeaux (km)")
ax.set_ylabel("price per m2 (EUR, log scale)")
ax.set_title("The gradient, raw and modelled: steep near the centre, flat far away")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "fig7_distance_gradient.png", dpi=150, bbox_inches="tight")

####  CHECK 复核5km断点表现

In [ ]:
# 【结论助手】把系数读成百分比（level 用 exp(b)−1，log 输入用 1.1^b−1），供第 8 步复用。
def _pct(fit, term, kind):
    # 把系数读成百分比变化及其 95 % 区间；kind='level' 用 exp(b)−1，kind='10pct' 用 1.1^b−1。
    b = float(fit.params[term])
    lo, hi = (float(v) for v in fit.conf_int().loc[term].iloc[:2])
    if kind == "10pct":
        return 100 * (1.1 ** b - 1), 100 * (1.1 ** lo - 1), 100 * (1.1 ** hi - 1)
    return 100 * (np.exp(b) - 1), 100 * (np.exp(lo) - 1), 100 * (np.exp(hi) - 1)


ROOM = _pct(h, "nombre_pieces_principales", "level")
HOUSEC = _pct(h, "C(type_local)[T.Maison]", "level")
LANDF = _pct(h, "has_land", "level")
LAND10 = _pct(h, "log_land", "10pct")

RES = {
    "n_sales": int(len(dvf)),
    "n_transactions": int(dvf_raw["id_mutation"].nunique()),
    "med_price": float(MED_PRICE),
    "med_surface": float(MED_SURFACE),
    "med_pm2_flat": float(dvf.loc[dvf["type_local"].eq("Appartement"), "prix_m2"].median()),
    "med_pm2_house": float(dvf.loc[dvf["type_local"].eq("Maison"), "prix_m2"].median()),
    "eur_10m2_flat": float(ten_m2.set_index("type").loc["Appartement", "EUR_per_10m2"]),
    "eur_10m2_flat_lo": float(ten_m2.set_index("type").loc["Appartement", "EUR_lo"]),
    "eur_10m2_flat_hi": float(ten_m2.set_index("type").loc["Appartement", "EUR_hi"]),
    "pct_10m2_flat": float(ten_m2.set_index("type").loc["Appartement", "pct_per_10m2"]),
    "pct_10m2_flat_lo": float(ten_m2.set_index("type").loc["Appartement", "pct_lo"]),
    "pct_10m2_flat_hi": float(ten_m2.set_index("type").loc["Appartement", "pct_hi"]),
    "eur_10m2_house": float(ten_m2.set_index("type").loc["Maison", "EUR_per_10m2"]),
    "eur_10m2_house_lo": float(ten_m2.set_index("type").loc["Maison", "EUR_lo"]),
    "eur_10m2_house_hi": float(ten_m2.set_index("type").loc["Maison", "EUR_hi"]),
    "pct_10m2_house": float(ten_m2.set_index("type").loc["Maison", "pct_per_10m2"]),
    "pct_10m2_house_lo": float(ten_m2.set_index("type").loc["Maison", "pct_lo"]),
    "pct_10m2_house_hi": float(ten_m2.set_index("type").loc["Maison", "pct_hi"]),
    "elasticity_flat": float(ten_m2.set_index("type").loc["Appartement", "elasticity"]),
    "elasticity_house": float(ten_m2.set_index("type").loc["Maison", "elasticity"]),
    "dist_pct_per_km": float(100 * (np.exp(b1.params["distance_km"]) - 1)),
    "dist_pct_lo": float(100 * (np.exp(b1.conf_int().loc["distance_km", 0]) - 1)),
    "dist_pct_hi": float(100 * (np.exp(b1.conf_int().loc["distance_km", 1]) - 1)),
    "dist_pct_per_km_at_1": float(100 * (np.exp(b2.params["distance_km"] + 2 * b2.params["distance_km_sq"]) - 1)),
    "dist_pct_per_km_at_5": float(100 * (np.exp(b2.params["distance_km"] + 2 * b2.params["distance_km_sq"] * 5) - 1)),
    "dist_pct_per_km_at_20": float(100 * (np.exp(b2.params["distance_km"] + 2 * b2.params["distance_km_sq"] * 20) - 1)),
    "dist_pct_under_5": float(inside),
    "dist_pct_over_5": float(out_pct),
    "p_hinge": float(b3.pvalues["km_beyond_5"]),
    "dist_turning_km": float(b2.params["distance_km"] / (-2 * b2.params["distance_km_sq"])),
    "r2_dwelling": float(h0.rsquared),
    "r2_distance": float(h1.rsquared),
    "r2_full": float(h2.rsquared),
    "r2_zones": float(h3.rsquared),
    "zone_bordeaux_pct": float(pct_coef(b4, "C(zone)[T.Bordeaux]")[0]),
    "zone_metropole_pct": float(pct_coef(b4, "C(zone)[T.Métropole]")[0]),
    "h_10m2_pct": float(p10),
    "h_10m2_lo": float(p10_lo),
    "h_10m2_hi": float(p10_hi),
    "h_room_pct": float(100 * (np.exp(h.params["nombre_pieces_principales"]) - 1)),
    "h_house_pct": float(100 * (np.exp(h.params["C(type_local)[T.Maison]"]) - 1)),
    "h_land_pct": float(100 * (np.exp(h.params["log_land"]) - 1)),
    "h_land_flag_pct": float(100 * (np.exp(h.params["has_land"]) - 1)),
    "h_km_pct": float(100 * (np.exp(h.params["distance_km"]) - 1)),
    "h_surface_elast": float(h.params["log_surface"]),
    "h_surface_10pct": float(100 * (1.1 ** h.params["log_surface"] - 1)),
    "h_surface_10pct_lo": float(100 * (1.1 ** h.conf_int().loc["log_surface"].iloc[0] - 1)),
    "h_surface_10pct_hi": float(100 * (1.1 ** h.conf_int().loc["log_surface"].iloc[1] - 1)),
    "h_room_lo": float(ROOM[1]), "h_room_hi": float(ROOM[2]),
    "h_house_lo": float(HOUSEC[1]), "h_house_hi": float(HOUSEC[2]),
    "h_land_flag_lo": float(LANDF[1]), "h_land_flag_hi": float(LANDF[2]),
    "h_land10_pct": float(LAND10[0]), "h_land10_lo": float(LAND10[1]), "h_land10_hi": float(LAND10[2]),
    "p_room": float(h.pvalues["nombre_pieces_principales"]),
    "p_house": float(h.pvalues["C(type_local)[T.Maison]"]),
    "p_land_flag": float(h.pvalues["has_land"]),
    "p_km": float(h.pvalues["distance_km"]),
    "p_land10": float(h.pvalues["log_land"]),
    "p_dist_b1": float(b1.pvalues["distance_km"]),
    "n_ml": 0,
}
print("figures/key_results.json will be written at the end of step 7; keys kept:", len(RES))

#### Addition 阿卡雄湾与拉卡诺海滨的价格分布

In [ ]:
# 【复用 TP3C + 第六讲】5.5：均值区间 + Welch 检验 + Mann-Whitney 检验，复核 5 km 断点。
# 【复用 TP3C】几乎照搬：均值区间、Welch 检验、非参数检验。
G = dvf.assign(side=np.where(dvf["distance_km"] < 5, "inside 5 km", "beyond 5 km"))

rows = []
for side, sub in G.groupby("side"):
    pm2 = sub["prix_m2"]
    lo, hi = sm.stats.DescrStatsW(pm2).tconfint_mean()  # the course's interval of the MEAN
    rows.append({"side": side, "n": len(pm2), "mean": pm2.mean(), "ci_low": lo, "ci_high": hi,
                 "median": pm2.median()})
by_side = pd.DataFrame(rows)
print("price per m2, the two sides of the 5 km hinge")
print(by_side.round(0).to_string(index=False))

inside = G.loc[G["side"].eq("inside 5 km"), "prix_m2"]
outside = G.loc[G["side"].eq("beyond 5 km"), "prix_m2"]
welch = stats.ttest_ind(inside, outside, equal_var=False)  # Welch: unequal variances
mw = stats.mannwhitneyu(inside, outside, alternative="two-sided")  # no normality assumed
diff = inside.mean() - outside.mean()

print(f"\nWelch          t = {welch.statistic:6.1f}   p = {welch.pvalue:.1e}   (df = {welch.df:.0f})")
print(f"mean EUR/m2    inside {inside.mean():,.0f}   beyond {outside.mean():,.0f}   "
      f"-> {diff:+,.0f} EUR  ({100 * diff / outside.mean():+.1f} %)")
print(f"Mann-Whitney   U = {mw.statistic:8.0f}   p = {mw.pvalue:.1e}")
print(f"median EUR/m2  inside {inside.median():,.0f}   beyond {outside.median():,.0f}")

# 三种工具给出同一答案：两侧不是同一个市场，差距很大。
assert welch.pvalue < 1e-10 and mw.pvalue < 1e-10 and diff > 0

In [ ]:
# 【复用 TP2A + 第六讲】5.6：两个度假地（阿卡雄港、Lacanau-Océan 海滨）的距离，海岸溢价与图 8。
# 【复用 TP2A】同一条 haversine 再算两次：省的兩個度假目的地，阿卡雄湾
# 阿卡雄（镇与港口）与 Lacanau-Océan（海滨浴场）——波尔多家庭周末开去的两条海岸
# 两地离佩贝朗广场都约 50 km。
RESORTS = {"Arcachon": (44.6586, -1.1682), "Lacanau-Ocean": (44.9994, -1.1989)}
for name, (la, lo) in RESORTS.items():
    dvf["d_" + name] = haversine_km(dvf["latitude"].to_numpy(), dvf["longitude"].to_numpy(), la, lo)
dvf["d_resort"] = dvf[["d_Arcachon", "d_Lacanau-Ocean"]].min(axis=1)
dvf["near_resort"] = (dvf["d_resort"] < 5).astype(int)

# （1）到波尔多的距离相同，却是两个市场
far = dvf[dvf["distance_km"] >= 40].copy()
far["where"] = np.where(far["near_resort"].eq(1), "within 5 km of a resort", "elsewhere, 40 km +")
print("sales 40 km or more from Bordeaux")
print(far.groupby("where")["prix_m2"].agg(["size", "median"]).round(0).to_string())

# （2）每个度假地的圈层，以及该圈层的波尔多距离
rows = []
for name in RESORTS:
    col = "d_" + name
    for lo_, hi_ in [(0, 2), (2, 5), (5, 10), (10, 20), (20, 300)]:
        sub = dvf[(dvf[col] >= lo_) & (dvf[col] < hi_)]
        if len(sub) >= 20:
            rows.append({"resort": name, "ring_km": f"{lo_}-{hi_}", "n": len(sub),
                         "median_pm2": round(sub["prix_m2"].median()),
                         "median_bordeaux_km": round(sub["distance_km"].median())})
rings = pd.DataFrame(rows)
print("\nmedian EUR/m2 by ring around each resort")
print(rings.to_string(index=False))

# （3）回归：把波尔多梯度控制住，海岸溢价还在吗？
CTRL_R = "log_surface + nombre_pieces_principales + C(type_local) + log_land + has_land"
# dml 是在这些列出现之前复制的：按行把新列接回去。
d6 = dml.join(dvf[["d_Arcachon", "d_Lacanau-Ocean", "d_resort", "near_resort"]])
b5 = smf.ols(f"log_prix_m2 ~ {CTRL_R} + distance_km + near_resort", data=d6).fit()
b6 = smf.ols(f"log_prix_m2 ~ {CTRL_R} + distance_km + d_resort", data=d6).fit()
print(f"\nB5  within 5 km of a resort: {100 * (np.exp(b5.params['near_resort']) - 1):+.1f} %"
      f"   [p = {b5.pvalues['near_resort']:.0e}]   R2 = {b5.rsquared:.3f}")
print(f"    distance to Bordeaux in B5: {100 * (np.exp(b5.params['distance_km']) - 1):+.2f} %/km")
print(f"B6  each km further from the nearest resort: {100 * (np.exp(b6.params['d_resort']) - 1):+.2f} %"
      f"   R2 = {b6.rsquared:.3f}")
print(f"    distance to Bordeaux in B6: {100 * (np.exp(b6.params['distance_km']) - 1):+.2f} %/km")

# （4）图 8：价格随远离度假地而下降，而两条海岸的降速并不相同
fig, ax = plt.subplots(figsize=(7.6, 4.2))
for name, colour in [("Arcachon", "C0"), ("Lacanau-Ocean", "C1")]:
    part = rings[rings["resort"].eq(name)]
    x = [0.5, 3, 7, 14, 40][:len(part)]
    ax.plot(x, part["median_pm2"], marker="o", color=colour, label=name)
ax.set(xscale="log", xlabel="distance to the resort (km, log scale)", ylabel="median EUR/m2",
       title="Away from the coast: the resort premium fades over a few kilometres")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "fig8_resort_profile.png", dpi=110, bbox_inches="tight")
plt.show()

assert rings["n"].sum() > 1000 and b5.pvalues["near_resort"] < 1e-20 and b6.rsquared > b5.rsquared - 0.01

## 6. 可直接用于机器学习的数据集
| **观测** | 一套住宅的一笔简单交易（17,912 行，每个 `id_mutation` 一行） |
| **目标** | `log_prix = log(valeur_fonciere)`——取对数让误差成为相对误差，这是价格模型需要的，也与第 5 步的模型一致 |
| **特征** | `log_surface`、`nombre_pieces_principales`、`is_maison`、`has_land`、`log_land`、`distance_km`、`distance_km_sq`、`month` |
| **划分** | 随机 80 / 20，`random_state=0`——文件只覆盖一年，没有时间顺序需要尊重 |
| **标准化** | 均值与标准差**只在训练集**上计算，再同时应用到两边 |
| **导出** | `GroupNN_B_gironde_ml.csv` + `GroupNN_B_gironde_data_dictionary.csv` |

`zone` 作为元数据导出（可用于画图或稳健性检验），**不是特征**：方案把地址放在“距离”里，而 5.4 已经表明
距离在场之后行政区加不了多少。

In [ ]:
# 【复用 TP4】第 6 步：构造 ML-ready 表——目标、8 个特征、随机 80/20 划分，标准化只在训练集上算。
ml = dvf.dropna(subset=["nombre_pieces_principales"]).copy()
FEATURES = ["log_surface", "nombre_pieces_principales", "is_maison", "has_land", "log_land",
            "distance_km", "distance_km_sq", "month"]
TARGET = "log_prix"

ml["split"] = np.where(np.random.default_rng(0).random(len(ml)) < 0.8, "train", "test")
train = ml[ml["split"].eq("train")]

# 【复用 TP4】标准化统计量只来自训练集：测试集从不参与均值和标准差的计算。
SCALER = pd.DataFrame({
    "column": FEATURES,
    "train_mean": [train[c].mean() for c in FEATURES],
    "train_std": [train[c].std(ddof=0) for c in FEATURES],
})
for c in FEATURES:
    ml["z_" + c] = (ml[c] - train[c].mean()) / train[c].std(ddof=0)

print(f"{len(ml):,} sales -> train {train.shape[0]:,} ({train.shape[0] / len(ml):.0%}) / test {len(ml) - len(train):,}")
print(f"target {TARGET}: train mean {train[TARGET].mean():.3f} | train std {train[TARGET].std(ddof=0):.3f}")
print(SCALER.round(3).to_string(index=False))
# 按构造，标准化以训练集为中心，而不是测试集。
assert abs(ml.loc[ml["split"].eq("train"), "z_" + FEATURES[0]].mean()) < 1e-9
assert abs(ml.loc[ml["split"].eq("test"), "z_" + FEATURES[0]].mean()) > 1e-3
assert ml.loc[ml["split"].eq("train"), TARGET].notna().all() and ml.loc[ml["split"].eq("test"), TARGET].notna().all()

In [ ]:
# 【复用 TP4】泄漏对照：全表标准化 vs 训练集标准化，测试集 z 最多差多少。
# 【复用 TP4 第三部分】经典泄漏：用全表的均值与标准差做标准化。上面的 z 列用的是
# 只用训练集统计量。这里看差别有多大，以及为什么在这个项目里几乎看不见。
x = "log_surface"
leaky = (ml[x] - ml[x].mean()) / ml[x].std(ddof=0)  # computed on train + test: a leak
ours = ml["z_" + x]
test_mask = ml["split"].eq("test")
gap = (leaky[test_mask] - ours[test_mask]).abs().max()
print(f"test rows: max |leaky z - our z| = {gap:.3f} standard deviations")
print("Under a random 80/20 split a leak is almost invisible — which is exactly why the split is STORED in a\n" +
      "column instead of being redrawn later: split by date, or by commune, and the same mistake becomes real.")
assert gap < 0.05

In [ ]:
# 【复用 TP4】导出 CSV：列名去重，并断言没有重复列。
EXPORT = list(dict.fromkeys(
    ["id_mutation", "split", "date_mutation", "month", "nom_commune", "code_commune", "zone",
     "type_local", "latitude", "longitude", "surface_reelle_bati", "nombre_pieces_principales",
     "surface_terrain_total", "n_dependances", "valeur_fonciere", "prix_m2", "distance_km", TARGET] + FEATURES
    + ["z_" + c for c in FEATURES]))  # dict.fromkeys: a feature that is also a raw column is exported once
export = ml[EXPORT].sort_values("split", ascending=False)  # train block first, then test
assert export.columns.is_unique, "the export must not repeat a column name"
assert len(export.columns) == len(EXPORT)
export.to_csv("GroupNN_B_gironde_ml.csv", index=False)
print(f"written GroupNN_B_gironde_ml.csv  {export.shape[0]:,} rows x {export.shape[1]} columns")
export.head(3)

In [ ]:
# 【复用 TP4】导出 31 行数据字典（类型 / 单位 / 含义 / 怎么构造）。
types = {
    "id_mutation": "string", "split": "string", "date_mutation": "date", "month": "integer 1-12",
    "nom_commune": "string", "code_commune": "string (INSEE)", "zone": "category (metadata)",
    "type_local": "category", "latitude": "float (degree)", "longitude": "float (degree)",
    "surface_reelle_bati": "float (m2)", "nombre_pieces_principales": "float (rooms)",
    "surface_terrain_total": "float (m2)", "n_dependances": "integer (count)",
    "valeur_fonciere": "float (EUR)", "prix_m2": "float (EUR/m2)", "distance_km": "float (km)",
    "log_prix": "float (log EUR)", "log_surface": "float (log m2)", "log_land": "float (log m2)",
    "is_maison": "integer 0/1", "has_land": "integer 0/1", "distance_km_sq": "float (km2)",
}
units = {
    "distance_km": "km", "distance_km_sq": "km2", "log_prix": "log(EUR)", "log_surface": "log(m2)",
    "log_land": "log(m2)", "is_maison": "0/1", "has_land": "0/1", "surface_terrain_total": "m2",
    "nombre_pieces_principales": "rooms", "n_dependances": "count", "month": "month (1-12)",
}
meaning = {
    "id_mutation": "identifier of the sale; one row per sale, unique",
    "split": "train (80 %) or test (20 %), random_state=0; the test set was never used to estimate anything",
    "date_mutation": "date of the sale, 2024",
    "month": "month of the sale, 1-12, kept because the number of sales is seasonal",
    "nom_commune": "commune where the dwelling is, from the file",
    "code_commune": "INSEE code of the commune (stable key)",
    "zone": "Bordeaux | Metropole | Reste de la Gironde, metadata only (not a feature)",
    "type_local": "Maison (house) or Appartement (flat), the only two types kept",
    "latitude": "latitude of the sale, WGS84",
    "longitude": "longitude of the sale, WGS84",
    "surface_reelle_bati": "surface of the dwelling, m2, DVF field, after removing 0 and impossible values",
    "nombre_pieces_principales": "number of main rooms, DVF field; 0 replaced by missing, 9 sales dropped",
    "surface_terrain_total": "land sold in the sale, m2: largest value per (parcel, land category), summed over the sale",
    "n_dependances": "number of outbuildings sold with the dwelling (garage, shed, ...)",
    "valeur_fonciere": "price of the whole sale, EUR; repeated on every row of the raw file, taken once here",
    "prix_m2": "valeur_fonciere / surface_reelle_bati, kept between 300 and 20,000",
    "distance_km": "great-circle distance from the sale to Place Pey Berland (centre of Bordeaux)",
    "distance_km_sq": "distance_km squared, the quadratic term of sub-question 2",
    "log_prix": "TARGET: log(valeur_fonciere), the price the models explain",
    "log_surface": "log(surface_reelle_bati)",
    "is_maison": "1 if a house, 0 if a flat",
    "has_land": "1 if the sale records a land area, 0 otherwise (most flats have none)",
    "log_land": "log(1 + land m2), 0 when no land is recorded",
}
how = {
    "id_mutation": "kept from the raw file",
    "split": "random 80/20 on the rows (one year of data, no time order)",
    "date_mutation": "kept, parsed as a date",
    "month": "date.dt.month",
    "nom_commune": "kept, taken on the dwelling row",
    "code_commune": "kept, the zone is built from it",
    "zone": "from code_commune (33063 = Bordeaux, the 28 INSEE codes of the Metropole, else the rest)",
    "type_local": "kept on the dwelling row",
    "latitude": "kept, rows outside the Gironde box removed",
    "longitude": "kept, rows outside the Gironde box removed",
    "surface_reelle_bati": "raw field; rows with 0 or missing dropped",
    "nombre_pieces_principales": "raw field; 0 replaced by NaN; 9 missing rows dropped for the ML table",
    "surface_terrain_total": "max per (id_mutation, id_parcelle, nature_culture), summed per sale",
    "n_dependances": "count of distinct outbuilding rows of the sale",
    "valeur_fonciere": "raw field, first value of the sale; < 1,000 or > 5,000,000 removed",
    "prix_m2": "computed, then rows outside [300, 20,000] removed (131 sales)",
    "distance_km": "haversine(latitude, longitude; 44.8378, -0.5792)",
    "distance_km_sq": "distance_km ** 2",
    "log_prix": "np.log(valeur_fonciere)",
    "log_surface": "np.log(surface_reelle_bati)",
    "is_maison": "type_local == 'Maison'",
    "has_land": "surface_terrain_total is not null",
    "log_land": "np.log1p(surface_terrain_total.fillna(0))",
}

rows = []
for c in export.columns:
    base = c[2:] if c.startswith("z_") else c
    rows.append({
        "column": c,
        "type": types.get(base, "float") + (" (z-score)" if c.startswith("z_") else ""),
        "unit": "standard deviations" if c.startswith("z_") else units.get(base, "-"),
        "meaning": f"z-score of {base}" if c.startswith("z_") else meaning.get(base, "-"),
        "how_built": f"({base} - training mean) / training std" if c.startswith("z_") else how.get(base, "-"),
    })
dictionary = pd.DataFrame(rows)
unmapped = dictionary.loc[dictionary["how_built"].eq("-") & ~dictionary["column"].str.startswith("z_"), "column"]
if len(unmapped):
    print("WARNING: no description for", ", ".join(unmapped))
dictionary.to_csv("GroupNN_B_gironde_data_dictionary.csv", index=False)
print(f"written GroupNN_B_gironde_data_dictionary.csv  ({len(dictionary)} columns, {export.shape[1]} exported)")
dictionary.head(8)

## 7. 基线



In [ ]:
# 【复用 TP4】第 7 步：四条基线规则（含常数预测）+ metrics() 评估函数。
def metrics(pred_price, truth):
    err = (pred_price - truth["valeur_fonciere"]).abs()
    return {
        "MAE_price_EUR": err.mean(),
        "MAE_pm2_EUR": (pred_price / truth["surface_reelle_bati"] - truth["prix_m2"]).abs().mean(),
        "R2_log_target": 1 - ((truth[TARGET] - np.log(pred_price)) ** 2).sum()
                         / ((truth[TARGET] - truth[TARGET].mean()) ** 2).sum(),
        "median_abs_err_pct": 100 * (err / truth["valeur_fonciere"]).median(),
    }


train = ml[ml["split"].eq("train")]
test = ml[ml["split"].eq("test")]
global_med = train["prix_m2"].median()
commune_med = train.groupby("code_commune")["prix_m2"].median()

rule_global = pd.Series(global_med, index=test.index) * test["surface_reelle_bati"]
rule_commune = (test["code_commune"].map(commune_med).fillna(global_med) * test["surface_reelle_bati"])

# 第 5 步的特征价格模型，只在训练行上重新拟合，然后在测试行上评估。
train_ids = set(train["id_mutation"])
m_test = smf.ols(f"log_prix ~ {DWELLING} + distance_km + C(month)", data=dml[dml["id_mutation"].isin(train_ids)]).fit()
test_rows = dml[dml["id_mutation"].isin(test["id_mutation"])]
pred_ols = np.exp(m_test.predict(test_rows))

# 同一模型再加每个市镇一个虚拟变量：画不出的“本地水平”。
m_comm = smf.ols(f"log_prix ~ {DWELLING} + distance_km + C(month) + C(code_commune)",
                 data=dml[dml["id_mutation"].isin(train_ids)]).fit()
test_known = test_rows[test_rows["code_commune"].isin(train["code_commune"])]
pred_comm = np.exp(m_comm.predict(test_known))

share_known = float(test["code_commune"].isin(commune_med.index).mean())
# 【复用 TP4 第六部分】“准确率很高——但这个基线为什么无用？”它的回归版是
# 全零分类器的孪生：常数预测——对数价格的 R² = 0，而欧元误差
# 并不比全局规则差多少。
rule_constant = pd.Series(train["valeur_fonciere"].median(), index=test.index)

rows = [("the median price, for every sale (TP4's twin)", rule_constant, test, 0.0),
        ("global median", rule_global, test, 0.0),
        ("median of the commune", rule_commune, test, share_known),
        ("hedonic OLS (step 5)", pred_ols, test, 1.0),
        ("hedonic OLS + commune", pred_comm, test_known, 1.0)]
baseline = pd.DataFrame([{"rule": name,
                          **{k: metrics(pred, truth)[k] for k in ["MAE_price_EUR", "MAE_pm2_EUR",
                                                                  "R2_log_target", "median_abs_err_pct"]},
                          "n": len(truth), "share_communes_known": share} for name, pred, truth, share in rows]).round(
    1)
print(baseline.to_string(index=False))
print(f"\nthe median price of the test set is {test['valeur_fonciere'].median():,.0f} EUR")
print(
    f"(the last row uses the {len(test_known):,} test sales of {len(test_rows):,} whose commune is in the training set)")

In [ ]:
# 【复用 TP4】把基线结果写进 RES，供结论与 PDF 使用。
RES["baseline_global_mae"] = float(metrics(rule_global, test)["MAE_price_EUR"])
RES["baseline_commune_mae"] = float(metrics(rule_commune, test)["MAE_price_EUR"])
RES["baseline_commune_fixed_mae"] = float(metrics(pred_comm, test_known)["MAE_price_EUR"])
RES["ols_test_mae"] = float(metrics(pred_ols, test)["MAE_price_EUR"])
RES["ols_test_r2"] = float(metrics(pred_ols, test)["R2_log_target"])
RES["commune_fixed_r2"] = float(metrics(pred_comm, test_known)["R2_log_target"])
RES["n_ml"] = int(len(ml))
RES["share_communes_known"] = float(share_known)
print(pd.DataFrame([{"model": "hedonic OLS (step 5)", "R2_log_price_test": RES["ols_test_r2"]},
                    {"model": "hedonic OLS + commune", "R2_log_price_test": RES["commune_fixed_r2"]}])
      .round(3).to_string(index=False))

## 8. 结论与局限

*TODO：回答问题、局限、后续步骤。*

====================================================================================
ANSWER TO THE MAIN QUESTION
====================================================================================
In 2024, the Gironde recorded 17,912 simple sales of one dwelling (61%
of the 29,429 transactions of the file), at a median price of 233,238 EUR
for a median surface of 77 m2 (3,678 EUR/m2 for a flat, 3,152 EUR/m2 for a house).
(the models below use the 17,903 sales that also record a number of rooms).

What each characteristic is worth (hedonic OLS on the price of the sale, all else equal):
  +10 m2 at the median surface   +8.6 %   [+8.2; +8.9]
  one more room, same surface    +4.3 %
  a house rather than a flat     +3.0 %
  one km further from the centre -0.5 %
  the dwelling explains R2 = 0.37 of the log price, the distance adds 2.0 points,
  the month 0.06 points, and the three administrative zones 2.72 more:
  the city limit is not a kilometre — at the same distance, inside/outside Bordeaux still differ (B4).

SUB-QUESTION 1 -- 10 m2 more, at the median of each market (quantile regression + elasticity)
  flat  +28,616 EUR  [28,120; 29,111]  =  +14.8 %  [+14.4; +15.2]
  house +23,952 EUR  [23,160; 24,743]  =  +8.0 %  [+7.7; +8.3]
  the log-log slopes are 0.76 (flat) and 0.77 (house): a dwelling 10 % larger
  sells for 7.5 % more (flat) and 7.6 % more (house) — the price grows less than the surface.
  The controlled effect (+10 m2 at the median surface, step 5.4) is +8.6 % — the number to quote as 'all else equal'.

SUB-QUESTION 2 -- the distance gradient (per m2, dwelling controlled)
  one km further, all else equal: -0.50 %   [-0.54; -0.46]   (linear B1)
  the gradient is not constant (B2): -1.76 %/km at 1 km, -1.60 at 5 km,
  -0.97 at 20 km — the parabola turns up at 43 km, so read the dots, not the curve beyond that.
  the hinge at 5 km (B3): -8.91 %/km under 5 km, -0.25 %/km beyond;
  the change is +8.66 points of %/km, p = 9.2e-156:
  yes, the gradient weakens sharply beyond 5 km — the first kilometres of Bordeaux
  carry the price, the rest of the department is almost flat (about 0.2 %/km).

BASELINE (test set, 20 %)
  global median rule          MAE 113,996 EUR
  commune median rule         MAE 76,373 EUR   (99.6% of the test in a commune known from the train)
  hedonic OLS (step 5)        MAE 105,428 EUR   R2 on the log price 0.384
  hedonic OLS + commune       MAE 73,230 EUR   R2 0.682   <- the bar for semester 2

LIMITS
- One year only (2024): no trend, and the 2024 rate context is not a normal market.
- Simple sales only: 17,912 of 29,429 transactions. Multi-dwelling sales,
  new builds (VEFA), exchanges and auctions are excluded: the sample is the comparable market, not all of it.
- The distance is a great circle to Place Pey Berland: it ignores the road, the bridges, the traffic and
  the public transport. It is a proxy for access, not a measure of it (no time, no mode).
- The centre is a decision: moving it a few hundred metres changes the intercept more than the slope, but the
  '1 km' of the model is not the '1 km' of a commuter.
- One gradient for the whole department: the coast (Arcachon basin) is far in km but expensive (Figure 5), so
  the distance coefficient is an average and hides a local premium the model cannot see.
- A continuous distance cannot draw a city limit: at the same kilometre, inside and outside Bordeaux still
  differ by about 46 % per m2 (B4). A boundary dummy would improve any next model;
  it is a decision (which boundary?) and not a natural fact.
- No information on floor, orientation, condition, renovation, view or noise: the R2 is capped by the data.
- A random 80/20 split shares communes between the two sets. Predicting a sale in a commune never seen
  (the real use case of a small agency) would be harder: that is the honest next test.

NEXT STEPS
- Add 2023 and 2025: a time split, a trend instead of one year, and a real out-of-sample test.
- Model the multi-dwelling sales with a per-lot price allocation, instead of dropping them.
- Replace the distance by travel time (public transport / car) and by the amenities that the centre gives
  access to: the part of the R2 that the kilometre cannot explain is the next question.

# 主要研究问题的回答

2024年，吉伦特省记录了 **17,912笔仅涉及一套住宅的简单买卖交易**，占文件中29,429笔交易的61%。成交价格中位数为 **233,238欧元**，住宅面积中位数为 **77平方米**。公寓的每平方米价格中位数为3,678欧元，独立屋为3,152欧元。

以下模型使用其中同时记录了房间数量的17,903笔交易。

## 各项特征值多少钱？

根据以成交总价为研究对象的享乐价格普通最小二乘回归（OLS），在其他条件不变时：

| 特征变化 | 成交价格变化 |
|---|---:|
| 在面积中位数处增加10平方米 | +8.6%，95%置信区间为［+8.2%；+8.9%］ |
| 面积不变，增加一个房间 | +4.3% |
| 独立屋相对于公寓 | +3.0% |
| 距市中心每远一公里 | −0.5% |

住宅自身特征对成交价格对数的解释度为 **R² = 0.37**。加入其他变量后，解释度的增量分别为：

- 距离变量：提高2.0个百分点；
- 月份变量：提高0.06个百分点；
- 三个行政分区变量：再提高2.72个百分点。

**城市边界的作用无法简单归结为距离：即使距市中心相同，位于波尔多市界内外的住宅价格仍存在差异（B4）。**

# 子问题1：在各自市场的面积中位数处，增加10平方米值多少钱？

采用分位数回归与弹性分析，结果如下：

| 住宅类型 | 增加10平方米对应的价格增量 | 百分比增幅 |
|---|---:|---:|
| 公寓 | +28,616欧元，95%置信区间为［28,120；29,111］欧元 | +14.8%，95%置信区间为［+14.4%；+15.2%］ |
| 独立屋 | +23,952欧元，95%置信区间为［23,160；24,743］欧元 | +8.0%，95%置信区间为［+7.7%；+8.3%］ |

双对数模型的斜率分别为：

- 公寓：0.76；
- 独立屋：0.77。

面积大10%的住宅，成交价格分别高约7.5%和7.6%。也就是说，**价格的增长幅度小于面积的增长幅度**。

> 控制其他变量后，在面积中位数处增加10平方米的价格效应为 **+8.6%**（第5.4步）。如果要引用“其他条件不变时”的结果，应使用这一数值。

# 子问题2：距离的价格梯度

以下结果以每平方米价格为研究对象，并控制住宅自身特征。

## 线性模型B1

在其他条件不变时，距市中心每远一公里，每平方米价格下降 **0.50%**，95%置信区间为［−0.54%；−0.46%］。

## 二次项模型B2

价格梯度并非常数：

| 距市中心的距离 | 每增加一公里对应的价格变化 |
|---|---:|
| 1公里 | −1.76% |
| 5公里 | −1.60% |
| 20公里 | −0.97% |

二次曲线在约43公里处转而向上，因此超过这一距离后，应以散点所反映的实际分布为依据，而不要依赖曲线的走势。

## 在5公里处设置折点的分段线性模型B3

| 距离范围 | 每增加一公里对应的价格变化 |
|---|---:|
| 5公里以内 | −8.91% |
| 5公里以外 | −0.25% |

梯度变化为每公里 **+8.66个百分点**，p值为 **9.2 × 10⁻¹⁵⁶**。

**距离超过5公里后，价格梯度显著减弱。** 波尔多中心周边最初几公里的区位对价格起着关键作用，而省内其余地区的距离梯度接近平坦，约为每公里下降0.2%。

# 基准模型表现

以下结果基于占全部样本20%的测试集：

| 方法 | 平均绝对误差（MAE） | 成交价格对数的R² |
|---|---:|---:|
| 全局价格中位数预测法 | 113,996欧元 | — |
| 市镇价格中位数预测法 | 76,373欧元 | — |
| 享乐价格OLS模型（第5步） | 105,428欧元 | 0.384 |
| 享乐价格OLS模型＋市镇变量 | **73,230欧元** | **0.682** |

测试集中99.6%的交易所在市镇，均已在训练集中出现。

> **“享乐价格OLS模型＋市镇变量”的表现，是第二学期需要对照和超越的基准。**

# 局限性

- **只使用2024年一年的数据：**无法分析时间趋势，而且2024年的利率环境并不代表常态市场。

- **只研究简单买卖交易：**29,429笔交易中，仅保留17,912笔。多住宅交易、期房销售（VEFA）、交换和拍卖均被排除，因此样本代表的是可比住宅交易市场，而非整个市场。

- **距离采用到佩贝朗广场（Place Pey Berland）的球面大圆距离：**没有考虑道路、桥梁、交通拥堵和公共交通。它是可达性的代理变量，而非直接度量，因为它既不包含出行时间，也不区分交通方式。

- **市中心的位置是一项人为设定：**将中心移动几百米，对截距的影响通常大于对斜率的影响，但模型中的“一公里”并不等同于通勤者实际体验的“一公里”。

- **整个省共用一个距离梯度：**沿海地区，尤其是阿卡雄湾，虽然距离市中心较远，价格却较高（图5）。因此，距离系数只是平均效应，掩盖了模型无法识别的局部区位溢价。

- **连续距离变量无法刻画城市边界：**在距离相同的情况下，波尔多市界内外的每平方米价格仍相差约46%（B4）。在后续模型中加入边界虚拟变量，有助于改善模型；但采用哪条边界是一项研究选择，而非天然既定的事实。

- **缺少住宅品质信息：**数据不包含楼层、朝向、房屋状况、翻新情况、景观或噪声等信息，因此模型解释度受到数据本身的限制。

- **随机80/20划分使训练集与测试集共享部分市镇：**预测一个从未在训练集中出现过的市镇中的交易会更加困难，而这正是小型房产中介可能面临的实际应用场景。这应当成为下一步更严格、更贴近实际的检验。

# 后续工作

1. **加入2023年和2025年的数据：**按时间划分训练集和测试集，从单一年份分析扩展到趋势分析，并开展真正的样本外检验。

2. **对多住宅交易建模：**按各住宅单元分摊成交总价，而不是直接剔除这类交易。

3. **用出行时间和配套设施替代距离：**纳入公共交通或驾车出行时间，以及市中心所提供的各类便利设施的可达性。距离变量无法解释的那部分价格差异，将成为下一步的研究问题。
```